In [3]:
import pyperclip
import os
from pathlib import Path

# Global list of supported file extensions
SUPPORTED_EXTENSIONS = ['.cpp', '.h', '.txt', '.py']

# Global list of relative paths to filter out (directories to skip)
FILTERED_DIRECTORIES = ['.git', 'build']

# Global list of relative paths to filter out (files to skip)
FILTERED_FILES = []

# Global list of directories to filter out anywhere in the dtree
GLOBAL_FILTERED_DIRECTORIES = ['__pycache__']

# Global list of files to filter out anywhere in the tree
GLOBAL_FILTERED_FILES = ['__init__.py', 'todo.txt','CMakeCache.txt']

def read_supported_files(directory):
    """
    Reads all supported source files from a directory tree and formats them as a string.

    :param directory: The root directory to start reading from.
    :return: A formatted string containing the directory structure and file contents.
    """
    result = []

    # Function to format the directory structure overview using ASCII art
    def format_directory_structure(path, indent=0, prefix=""):
        relative_path = path.relative_to(directory)
        if any(filtered_dir in relative_path.parts for filtered_dir in FILTERED_DIRECTORIES) or path.name in GLOBAL_FILTERED_DIRECTORIES:
            return
        result.append(' ' * indent + prefix + os.path.basename(path) + '/')
        entries = sorted(path.iterdir())
        for index, item in enumerate(entries):
            is_last = index == len(entries) - 1
            new_prefix = "└── " if is_last else "├── "
            new_indent = indent + 4 if is_last else indent + 4
            if item.is_dir():
                format_directory_structure(item, new_indent, new_prefix)
            elif item.is_file() and item.suffix in SUPPORTED_EXTENSIONS and str(item.relative_to(directory)) not in FILTERED_FILES and item.name not in GLOBAL_FILTERED_FILES:
                result.append(' ' * new_indent + new_prefix + item.name)
        if indent==0:
            result.append('\n')

    # Function to read and format the content of supported files
    def format_supported_files(path):
        relative_path = path.relative_to(directory)
        if any(filtered_dir in relative_path.parts for filtered_dir in FILTERED_DIRECTORIES) or path.name in GLOBAL_FILTERED_DIRECTORIES:
            return
        for item in sorted(path.iterdir()):
            if item.is_dir():
                format_supported_files(item)
            elif item.is_file() and item.suffix in SUPPORTED_EXTENSIONS and str(item.relative_to(directory)) not in FILTERED_FILES and item.name not in GLOBAL_FILTERED_FILES:
                result.append(f"\n## {item.relative_to(directory)}")
                result.append("``` ")
                with open(item, 'r', encoding='utf-8') as file:
                    result.append(file.read())
                result.append("```")

    # Start with the directory structure overview
    # result.append("[ directory structure overview ]")
    # format_directory_structure(Path(directory))

    result.append("This is my project's code:")

    # Then format the supported files
    format_supported_files(Path(directory))

    return '\n'.join(result)

# Example usage
directory = r'.'
formatted_output = read_supported_files(directory)
# pyperclip.copy(formatted_output)

In [4]:
# To make new parser

parts = [formatted_output,
         r"""Your task is to transpile this code into the equivalent JS code, without any missing features. 
         Output the complete code for all files. Also make a demo.html page that demonstrates it with XOR. 
         """,  

         #"""Output the full code for the files requiring update. Make sure not to accidentally delete code or features I did not ask to be deleted."""
         ]

prompt = '\n'.join(parts)
pyperclip.copy(prompt)